In [ ]:
#@title MyDriveのマウントとutilの実行

from google.colab import drive
drive.mount('/content/drive')

# %run '/content/drive/MyDrive/表のグラフ化_ver2/表作成プログラム/util.ipynb'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#@title モジュールのインポート と googleスプレッドシートを読み込むための認証

import pandas as pd
from google.colab import auth
from google.auth import default
from google.auth.transport.requests import Request
from googleapiclient.discovery import build
import gspread
import io
import requests

# 1. 認証
auth.authenticate_user()
creds, _ = default()
if not creds.valid:
    creds.refresh(Request())

# 2. Google Drive APIの設定
drive_service = build('drive', 'v3', credentials=creds)

In [ ]:
#@title 処理対象のgoogleスプレッドシートのフォルダ選択
folder_name_target = "2014_成績表_googleスプレッドシート" # @param ['20260331_予防接種A_処理後_googleスプレッドシート', '20260331_予防接種B_処理後_googleスプレッドシート', '2014_成績表_googleスプレッドシート', '20260411_血液検査_googleスプレッドシート'] {type:"string"}

print( f"選択された予防接種のタイプ：{folder_name_target}")

選択された予防接種のタイプ：2014_成績表_googleスプレッドシート


In [ ]:
#@title 統合googleスプレッドシートのフォルダ選択
folder_name_integrated = "2014_成績表_統合" # @param ['20260331_予防接種A_統合', '20260331_予防接種B_統合', '2014_成績表_統合', '20260411_血液検査_統合'] {type:"string"}

print( f"選択された予防接種のタイプ：{folder_name_integrated}")

選択された予防接種のタイプ：2014_成績表_統合


In [ ]:
#@title フォルダid、ファイルidを取得する関数：get_file_names(folder_name):

def get_file_names(folder_name):

    # 検索対象のフォルダ名
    print(f'フォルダー名: {folder_name} を探索')

    # フォルダを検索
    folder_query = f"name = '{folder_name}' and mimeType = 'application/vnd.google-apps.folder' and trashed = false"
    folder_results = drive_service.files().list(q=folder_query, fields="files(id)").execute()

    # 実行結果からフォルダリストを取り出す
    folders = folder_results.get('files', [])

    if not folders:
        raise Exception(f"フォルダ '{folder_name}' が見つかりませんでした。")

    folder_id = folders[0]['id']

    # フォルダ内の「すべてのスプレッドシート」をリストアップ
    file_query = f"'{folder_id}' in parents and mimeType = 'application/vnd.google-apps.spreadsheet' and trashed = false"
    file_results = drive_service.files().list(
        q=file_query,
        fields="files(id, name)",
        orderBy="name"
    ).execute()

    files = file_results.get('files', [])

    # 例外処理
    if not files:
        print("スプレッドシートが見つかりません。全ファイル形式を確認します...")
        all_files = drive_service.files().list(q=f"'{folder_id}' in parents", fields="files(name, mimeType)").execute().get('files', [])
        for f in all_files:
            print(f" - {f['name']} (Type: {f['mimeType']})")
        raise Exception("フォルダ内にGoogleスプレッドシート形式のファイルが存在しません。")

    print(f"\nフォルダ「{folder_name}」内のスプレッドシート一覧:")
    for i, f in enumerate(files):
        print(f"  {i+1}: {f['name']} (ID: {f['id']})")

    print()

    # files (リスト) と folder_id (文字列) を両方返す
    return files, folder_id

In [ ]:
#@title googleスプレッドシートを読み込む関数

def load_gsp_and_change_code(target_file):

    # 4. 最初のファイルを選択
    spreadsheet_id = target_file['id']
    print(f"\n---> 実行対象: {target_file['name']} を読み込みます。")

    # 5. 高速読み込み：CSVエクスポートURLを使ってcsv形式でダウンロードする
    url = f'https://docs.google.com/spreadsheets/d/{spreadsheet_id}/export?format=csv'
    headers = {'Authorization': f'Bearer {creds.token}'}
    res = requests.get(url, headers=headers)

    if res.status_code != 200:
        raise Exception(f"データのダウンロードに失敗しました。ステータスコード: {res.status_code}")

    # 6. DataFrame化
    # pd.read_csv()はファイルパスを指定して、その内容を受けとる。
    # res.contentはcsv形式のバイナリデータであるため、io.BytesIO()で疑似ファイルオブジェクトに変換してpd.read_csv()に渡す。
    df = pd.read_csv(io.BytesIO(res.content))

    return df

In [84]:
#@title 最新の統合googleスプレッドシートの読み込み

files, folder_id_integrated = get_file_names(folder_name_integrated)

# リストが空でないか確認してから、最後の要素を取得
if files:
    latest_item = files[-1]
    latest_id = latest_item['id']
    latest_name = latest_item['name']
else:
    print("リストは空です")

print(f'最後に更新された統合ファイル: {latest_name}')
print()

df_latest = load_gsp_and_change_code(latest_item)
print(f"フォルダ内の最新のgoogleスプレッドシートを読み込みました: 長さ：{len(df_latest)}")

display(df_latest)

フォルダー名: 2014_成績表_統合 を探索



フォルダ「2014_成績表_統合」内のスプレッドシート一覧:
  1: 2014_0成績表  (ID: 1sxyEXwc2xzl3pN_Qqr9UKBsttPzOk0tPU5msls6Ig6I)

最後に更新された統合ファイル: 2014_0成績表 


---> 実行対象: 2014_0成績表  を読み込みます。
フォルダ内の最新のgoogleスプレッドシートを読み込みました: 長さ：0


,学期,教科,国語（平均）,社会（平均）,数学（平均）,理科（平均）,英語（平均）,５教科合計（平均）,音楽（平均）,美術（平均）,保体（平均）,技家（平均）


In [85]:
folder_name_target

'2014_成績表_googleスプレッドシート'

In [95]:
#@title 追加するgoogleスプレッドシートのフォルダ内のファイルの読み込み

files, folder_id_target = get_file_names(folder_name_target)

# リストが空でないか確認してから、最後の要素を取得
if files:
    latest_item = files[-1]
    latest_id = latest_item['id']
    latest_name = latest_item['name']
else:
    print("リストは空です")

フォルダー名: 2014_成績表_googleスプレッドシート を探索

フォルダ「2014_成績表_googleスプレッドシート」内のスプレッドシート一覧:
  1: 2014_1_1成績表 (ID: 1rEOFZ7zHcScBcfMNeatzAUMUr2jSp69M5yCWzrexebA)
  2: 2014_1_2成績表 (ID: 1kc10zkh8JADwXFAzwgyYWUDmNnr1V_4nNf1b4QvmaWs)
  3: 2014_2_1成績表 (ID: 1UuS_MfV8qhq8qWVq-1qjLFePfWgdLl2w6l6A2vc-N4A)
  4: 2014_2_2成績表 (ID: 1QQAOtkLfGudShrCO0VMt7G0coKOipTl-CrXU1LHkdP4)



In [138]:
#@title 処理対象の番号を指定してdfとして読み込み

file_idx =0 # @param [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11] {type:"raw"}
df_temp = load_gsp_and_change_code(files[file_idx])
print(f"フォルダ内のgoogleスプレッドシートの読み込み完了")

display(df_temp)


---> 実行対象: 2014_1_1成績表 を読み込みます。
フォルダ内のgoogleスプレッドシートの読み込み完了


,学期,教科,国語（平均）,社会（平均）,数学（平均）,理科（平均）,英語（平均）,５教科合計（平均）,音楽（平均）,美術（平均）,保体（平均）,技家（平均）
0,2026_1学期中間テスト1_1,得点,82(77),79(71),79(55),78(62),79(67),397(330),—(—),—(—),—(—),—(—)


In [ ]:
sssaa

In [140]:
#@title 統合版に1月分追加

result = pd.concat([df_latest, df_temp], axis=0, ignore_index=True)
display(result)

,学期,教科,国語（平均）,社会（平均）,数学（平均）,理科（平均）,英語（平均）,５教科合計（平均）,音楽（平均）,美術（平均）,保体（平均）,技家（平均）
0,2026_1学期中間テスト1_1,得点,82(77),79(71),79(55),78(62),79(67),397(330),—(—),—(—),—(—),—(—)


In [122]:
#@title 保存

from datetime import datetime
import numpy as np  #

# エラーの原因（NaN）を0や空文字に置き換える
# result は下の工程で使う DataFrame の名前です
result = result.replace({np.nan: None})
# または、数値の0にしたい場合は result = result.fillna(0)

# 4. ファイル名の設定
now_time = datetime.now().strftime('%Y%m%d_%H%M%S')
file_name = f"統合_{now_time}"

# 2. 実行時の日時を取得（重複しているので片方でもOK）
now_time = datetime.now().strftime('%Y%m%d_%H%M%S')
file_name = f"統合_{now_time}"

# 3. gc (Google Client) の作成
gc = gspread.authorize(creds)

# 5. 指定したフォルダ内にスプレッドシートを作成
sh = gc.create(file_name, folder_id_integrated)

# 6. 書き込み
worksheet = sh.get_worksheet(0)
# NaNを処理した後のresultをリスト形式に変換して書き込む
data_to_write = [result.columns.values.tolist()] + result.values.tolist()
worksheet.update('A1', data_to_write)

print(f"保存完了: {sh.url}")

/tmp/ipykernel_22647/3119752243.py:29: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  worksheet.update('A1', data_to_write)


保存完了: https://docs.google.com/spreadsheets/d/1CY5DtSm7UCiGGIlUsFKQ1E-QxWbAODVg1phgYWxaV2g
